**Parte 2 - API de lluvias**

Grupo 3 - Corresponde la temporada de: 1 de enero al 31 de mayo de 2024

1. Con pd.read_html, lean la tabla de departamentos de Wikipedia y quédense con las columnas departamento y capital. 
Wikipedia rechaza pd.read_html(url) directo (error 403), por eso descargamos la página con requests y un User-Agent, y luego leemos sus tablas.

In [15]:
# Importa io, que permite tratar un texto como si fuera un archivo.
import io

# Importa time, que usaremos para hacer pausas entre pedidos.
import time

# Importa unicodedata, que usaremos para quitar tildes al comparar nombres.
import unicodedata

# Importa requests, que permite realizar solicitudes a páginas web y APIs.
import requests

# Importa pandas, que permite trabajar con tablas (DataFrames).
import pandas as pd

# Define una cabecera que identifica nuestra solicitud como proveniente de un navegador.
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Define las fechas de inicio y fin de la temporada del grupo 3.
INICIO, FIN = "2024-01-01", "2024-05-31"

# Indica la dirección de la página de Wikipedia con los departamentos.
url_wiki = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

# Descarga la página enviando la cabecera, con un tiempo máximo de espera de 30 segundos.
respuesta = requests.get(url_wiki, headers=HEADERS, timeout=30)

# Imprime el código de estado (200 significa que la descarga fue exitosa).
print("Código de estado:", respuesta.status_code)

# Lee todas las tablas de la página y las guarda en una lista de DataFrames.
tablas = pd.read_html(io.StringIO(respuesta.text))

# Imprime el número, el tamaño (filas, columnas) y las primeras columnas de cada tabla.
for i, t in enumerate(tablas):
    print(i, t.shape, list(t.columns)[:6])

Código de estado: 200
0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


Usamos la tabla 1 (departamentos actuales). La tabla 2 es de departamentos históricos.

In [16]:
# Toma la tabla 1 y se queda solo con las columnas Departamento y Capital.
deptos = tablas[1][["Departamento", "Capital"]].copy()

# Renombra las columnas en minúsculas, como pide el enunciado.
deptos.columns = ["departamento", "capital"]

# Imprime cuántas filas tiene la tabla.
print("Filas:", len(deptos))

# Muestra la tabla.
deptos

Filas: 24


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


2. Verificación: la tabla final debe tener 25 filas (los 24 departamentos y el Callao). Si falta alguno, agréguenlo y expliquen de dónde lo sacaron.

La tabla tiene 24 filas: falta el Callao, que Wikipedia lista aparte como Provincia Constitucional (tabla 3), con capital Callao. Lo agregamos desde esa tabla.     

In [17]:
# Muestra la tabla 3, de donde tomamos la capital del Callao.
print(tablas[3][["Provincia", "Capital"]])

# Crea una tabla de una fila con el Callao y su capital.
callao = pd.DataFrame({"departamento": ["Callao"], "capital": ["Callao"]})

# Une la fila del Callao a la tabla de departamentos.
deptos = pd.concat([deptos, callao], ignore_index=True)

# Ordena alfabéticamente por departamento y reinicia la numeración de filas.
deptos = deptos.sort_values("departamento").reset_index(drop=True)

# Imprime el número de filas.
print("Filas:", len(deptos))

# Detiene el notebook con un error si no hay exactamente 25 filas.
assert len(deptos) == 25

        Provincia Capital
0  Callao[nota 6]  Callao
1    Lima[nota 7]    Lima
Filas: 25


La capital de Lima aparece como "Huacho (de facto)", que la API no reconoce. Usamos Lima (la ciudad): concentra la mayor parte de la población del departamento y, como Huacho, está en la costa, casi sin lluvia.

In [18]:
# Muestra las capitales que tienen texto extra entre paréntesis o corchetes.
print(deptos[deptos["capital"].str.contains(r"\(|\[", regex=True)])

# Reemplaza "Huacho (de facto)" por "Lima".
deptos["capital"] = deptos["capital"].replace({"Huacho (de facto)": "Lima"})

# Elimina notas entre corchetes (por ejemplo, "[nota 6]") y espacios sobrantes.
deptos["capital"] = deptos["capital"].str.replace(r"\[.*?\]", "", regex=True).str.strip()

# Muestra la tabla final de departamentos y capitales.
deptos

   departamento            capital
13         Lima  Huacho (de facto)


,departamento,capital
0,Amazonas,Chachapoyas
1,Apurímac,Abancay
2,Arequipa,Arequipa
3,Ayacucho,Ayacucho
4,Cajamarca,Cajamarca
5,Callao,Callao
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


3. Para cada capital, obtengan su latitud y longitud con la API de geocodificación: https://geocoding-api.open-meteo.com/v1/search?name=Chachapoyas&count=10&language=es

La API puede devolver varias ciudades con el mismo nombre (por ejemplo, hay Trujillo en otros países), así que nos quedamos con el primer resultado con country_code == "PE".

In [19]:
# Define una función que busca una ciudad en la API de geocodificación.
def geocodificar(ciudad):

    # Indica la dirección de la API de geocodificación.
    url = "https://geocoding-api.open-meteo.com/v1/search"

    # Define los parámetros: nombre de la ciudad, hasta 10 resultados y en español.
    params = {"name": ciudad, "count": 10, "language": "es"}

    # Realiza la solicitud y obtiene la lista de resultados (vacía si no hay ninguno).
    resultados = requests.get(url, params=params, timeout=30).json().get("results", [])

    # Recorre los resultados y devuelve el primero que esté en el Perú.
    for r in resultados:
        if r.get("country_code") == "PE":
            return r["admin1"], r["latitude"], r["longitude"], len(resultados)

    # Si ningún resultado está en el Perú, devuelve valores vacíos.
    return None, None, None, len(resultados)

# Crea una lista vacía para guardar los resultados de cada capital.
filas = []

# Recorre cada departamento de la tabla.
for _, fila in deptos.iterrows():

    # Busca la capital en la API.
    admin1, lat, lon, n = geocodificar(fila["capital"])

    # Guarda el departamento, la capital, lo que devolvió la API y cuántos resultados hubo.
    filas.append({"departamento": fila["departamento"], "capital": fila["capital"],
                  "admin1": admin1, "latitud": lat, "longitud": lon, "n_resultados": n})

    # Hace una pausa de 1 segundo antes del siguiente pedido.
    time.sleep(1)

# Convierte la lista de resultados en una tabla.
geo = pd.DataFrame(filas)

# Muestra la tabla.
geo

,departamento,capital,admin1,latitud,longitud,n_resultados
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,4
1,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,1
2,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,9
3,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,10
4,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,10
5,Callao,Callao,Provincia Constitucional del Callao,-12.05162,-77.13452,10
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,5
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,3
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,5
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,10


4. Verificación: la API puede devolver varias ciudades con el mismo nombre, dentro y fuera del Perú. Impriman una tabla con departamento, capital, el departamento que devolvió la API (admin1), latitud y longitud. Todas las filas deben estar en el Perú y en el departamento correcto. Expliquen cómo eligieron el resultado correcto.

In [20]:
# Define una función que deja un nombre en minúsculas, sin prefijos y sin tildes.
def normalizar(texto):

    # Convierte el texto a minúsculas.
    texto = str(texto).lower()

    # Quita los prefijos que agrega la API.
    for prefijo in ["departamento de ", "provincia constitucional del ", "provincia de "]:
        texto = texto.replace(prefijo, "")

    # Separa cada letra de su tilde (por ejemplo, "á" pasa a "a" + tilde).
    texto = unicodedata.normalize("NFKD", texto)

    # Elimina las tildes y los espacios sobrantes.
    return "".join(c for c in texto if not unicodedata.combining(c)).strip()

# Crea una columna que indica si el departamento coincide con admin1 después de normalizar.
geo["admin1_ok"] = geo["departamento"].apply(normalizar) == geo["admin1"].apply(normalizar)

# Imprime la tabla de verificación completa.
print(geo[["departamento", "capital", "admin1", "latitud", "longitud", "admin1_ok"]].to_string())

# Cuenta cuántas capitales no tuvieron resultado en el Perú.
print("\nSin resultado en Perú:", geo["latitud"].isna().sum())

# Cuenta cuántas capitales quedaron en un departamento distinto.
print("admin1 distinto:", (~geo["admin1_ok"]).sum())

# Detiene el notebook con un error si alguna fila no pasa la verificación.
assert geo["latitud"].notna().all() and geo["admin1_ok"].all()

     departamento           capital                               admin1   latitud  longitud  admin1_ok
0        Amazonas       Chachapoyas             Departamento de Amazonas  -6.23169 -77.86903       True
1        Apurímac           Abancay                             Apurímac -13.63426 -72.88380       True
2        Arequipa          Arequipa             Departamento de Arequipa -16.39899 -71.53747       True
3        Ayacucho          Ayacucho             Departamento de Ayacucho -13.16380 -74.22345       True
4       Cajamarca         Cajamarca            Departamento de Cajamarca  -7.16378 -78.50027       True
5          Callao            Callao  Provincia Constitucional del Callao -12.05162 -77.13452       True
6           Cusco             Cusco                Departamento de Cusco -13.53188 -71.96701       True
7    Huancavelica      Huancavelica         Departamento de Huancavelica -12.78691 -74.97298       True
8         Huánuco           Huánuco              Departamento de

In [21]:
# Busca "Trujillo" en la API para mostrar que devuelve varias ciudades con el mismo nombre.
ejemplo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": "Trujillo", "count": 10, "language": "es"},
                       timeout=30).json()["results"]

# Muestra el país y el departamento (admin1) de cada resultado.
pd.DataFrame(ejemplo)[["name", "country_code", "admin1", "latitude", "longitude"]]

,name,country_code,admin1,latitude,longitude
0,Trujillo,PE,Departamento de La Libertad,-8.11599,-79.02998
1,Trujillo,VE,Estado Trujillo,9.36587,-70.43659
2,Trujillo,HN,Departamento de Colón,15.91741,-85.95263
3,Trujillo,ES,Comunidad Autónoma de Extremadura,39.45786,-5.88203
4,Trujillo,CO,Valle del Cauca,4.21217,-76.31945
5,Trujillo,MX,Estado de Zacatecas,23.15881,-103.18071
6,Trujillo,PE,Departamento de Loreto,-4.17610,-73.27747
7,Trujillo,MX,Estado de Jalisco,21.68361,-103.76972
8,Trujillo,PH,Bisayas Orientales,12.43861,124.52611
9,Trujillo,CU,Provincia de Cienfuegos,22.20942,-80.45971


Cómo elegimos el resultado correcto:

1. La API devuelve hasta 10 ciudades con el mismo nombre. Por ejemplo, "Trujillo" aparece en Venezuela, Honduras, España, Colombia, México, Filipinas y Cuba. Por eso recorrimos los resultados y nos quedamos con el primero con `country_code == "PE"`.
2. Eso no basta: "Trujillo" también existe dos veces en el Perú (La Libertad y Loreto). Por eso verificamos que el `admin1` devuelto coincida con el departamento.
3. `admin1` no viene escrito igual que nuestros nombres, así que comparamos después de quitar prefijos, tildes y mayúsculas. Revisamos fila por fila los casos distintos:
   - Áncash → "Ancash" (sin tilde)
   - Callao → "Provincia Constitucional del Callao"
   - Lima → "Provincia de Lima"
   - Apurímac, Puno, Ucayali → sin el prefijo "Departamento de"

Resultado: las 25 capitales están en el Perú y en su departamento correcto.

5. Para cada capital, pidan la lluvia diaria de su temporada con la API histórica: https://archive-api.open-meteo.com/v1/archive?latitude=-6.23&longitude=-77.87&start_date=2023-01-01&end_date=2023-05-31&daily=precipitation_sum&timezone=America/Lima
Hagan una pausa de al menos 1 segundo entre pedidos.

In [22]:
# Define una función que descarga la lluvia diaria de un punto (latitud, longitud).
def lluvia_diaria(lat, lon):

    # Indica la dirección de la API histórica.
    url = "https://archive-api.open-meteo.com/v1/archive"

    # Define los parámetros: coordenadas, fechas de la temporada, variable y zona horaria.
    params = {"latitude": lat, "longitude": lon, "start_date": INICIO, "end_date": FIN,
              "daily": "precipitation_sum", "timezone": "America/Lima"}

    # Realiza la solicitud y convierte la respuesta en un diccionario.
    datos = requests.get(url, params=params, timeout=30).json()

    # Devuelve una tabla con la fecha y la lluvia de cada día.
    return pd.DataFrame({"fecha": datos["daily"]["time"],
                         "precipitacion_mm": datos["daily"]["precipitation_sum"]})

# Crea una lista vacía para guardar la tabla de cada departamento.
diarios = []

# Recorre cada departamento con sus coordenadas.
for _, fila in geo.iterrows():

    # Descarga la lluvia diaria de la capital.
    df = lluvia_diaria(fila["latitud"], fila["longitud"])

    # Agrega el nombre del departamento a cada fila.
    df["departamento"] = fila["departamento"]

    # Guarda la tabla en la lista.
    diarios.append(df)

    # Hace una pausa de 1 segundo antes del siguiente pedido.
    time.sleep(1)

# Une las 25 tablas en una sola.
lluvia = pd.concat(diarios, ignore_index=True)

# Imprime el número total de filas.
print("Filas:", len(lluvia), "(25 departamentos × 152 días)")

# Muestra las primeras 5 filas de la tabla de lluvia diaria.
lluvia.head()

Filas: 3800 (25 departamentos × 152 días)


,fecha,precipitacion_mm,departamento
0,2024-01-01,0.7,Amazonas
1,2024-01-02,1.1,Amazonas
2,2024-01-03,2.3,Amazonas
3,2024-01-04,0.6,Amazonas
4,2024-01-05,0.3,Amazonas


6. Para cada departamento, calculen:
- lluvia_total_mm: la suma de la lluvia diaria de la temporada.
- dias_lluvia_fuerte: el número de días con 20 mm o más.

In [23]:
# Agrupa por departamento y calcula la lluvia total y los días con 20 mm o más.
resumen = lluvia.groupby("departamento").agg(
    lluvia_total_mm=("precipitacion_mm", "sum"),
    dias_lluvia_fuerte=("precipitacion_mm", lambda s: (s >= 20).sum()),
).reset_index()

# Redondea la lluvia total a un decimal.
resumen["lluvia_total_mm"] = resumen["lluvia_total_mm"].round(1)

# Une la tabla de coordenadas con el resumen de lluvia.
final = geo[["departamento", "capital", "latitud", "longitud"]].merge(resumen, on="departamento")

# Ordena de mayor a menor lluvia total.
final = final.sort_values("lluvia_total_mm", ascending=False).reset_index(drop=True)

# Muestra la tabla final.
final

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Loreto,Iquitos,-3.74814,-73.25290,1361.8,19
1,Junín,Huancayo,-12.06866,-75.21027,1276.9,11
2,Madre de Dios,Puerto Maldonado,-12.58946,-69.19948,1149.5,15
3,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3
4,Ucayali,Pucallpa,-8.37915,-74.55387,887.4,9
5,Pasco,Cerro de Pasco,-10.66577,-76.25309,839.3,1
6,Huancavelica,Huancavelica,-12.78691,-74.97298,819.2,1
7,Puno,Puno,-15.84003,-70.02198,728.6,1
8,San Martín,Moyobamba,-6.03441,-76.97423,681.7,3
9,Cajamarca,Cajamarca,-7.16378,-78.50027,671.2,0


Las capitales con más lluvia son Iquitos (Loreto), Huancayo (Junín) y Puerto Maldonado (Madre de Dios); las costeras no llegan a 30 mm en toda la temporada. Lima y Callao tienen el mismo valor porque están tan cerca que caen en la misma celda de la grilla de Open-Meteo. **Limitación:** medimos un solo punto por departamento (la capital), que no representa a todo el territorio.

7. Verificación: cuenten cuántos días llegaron sin dato (None) y expliquen qué hicieron con ellos. Si el resultado es 0, también es una respuesta válida: lo importante es haberlo comprobado.

In [24]:
# Cuenta los días sin dato (pandas convierte los None de la API en NaN).
print("Días sin dato:", lluvia["precipitacion_mm"].isna().sum())

# Muestra los días sin dato de cada departamento.
lluvia.groupby("departamento")["precipitacion_mm"].apply(lambda s: s.isna().sum())

Días sin dato: 0


departamento
Amazonas         0
Apurímac         0
Arequipa         0
Ayacucho         0
Cajamarca        0
Callao           0
Cusco            0
Huancavelica     0
Huánuco          0
Ica              0
Junín            0
La Libertad      0
Lambayeque       0
Lima             0
Loreto           0
Madre de Dios    0
Moquegua         0
Pasco            0
Piura            0
Puno             0
San Martín       0
Tacna            0
Tumbes           0
Ucayali          0
Áncash           0
Name: precipitacion_mm, dtype: int64

No hay días sin dato: los 25 departamentos tienen sus 152 días completos. Si los hubiera, sum() los ignoraría y subestimaría la lluvia, así que los reportaríamos como limitación en lugar de rellenarlos.

8. Guarden datos/lluvias_por_departamento.csv con las columnas departamento, capital, latitud, longitud, lluvia_total_mm y dias_lluvia_fuerte. 

In [25]:
# Comprueba que la tabla final tenga 25 filas.
assert len(final) == 25

# Guarda la tabla en CSV, con utf-8-sig para que las tildes se vean bien en Excel.
final.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")

# Lee el archivo guardado para comprobar que quedó bien.
pd.read_csv("datos/lluvias_por_departamento.csv")

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Loreto,Iquitos,-3.74814,-73.25290,1361.8,19
1,Junín,Huancayo,-12.06866,-75.21027,1276.9,11
2,Madre de Dios,Puerto Maldonado,-12.58946,-69.19948,1149.5,15
3,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3
4,Ucayali,Pucallpa,-8.37915,-74.55387,887.4,9
5,Pasco,Cerro de Pasco,-10.66577,-76.25309,839.3,1
6,Huancavelica,Huancavelica,-12.78691,-74.97298,819.2,1
7,Puno,Puno,-15.84003,-70.02198,728.6,1
8,San Martín,Moyobamba,-6.03441,-76.97423,681.7,3
9,Cajamarca,Cajamarca,-7.16378,-78.50027,671.2,0
